In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Imports loaded")

✅ Imports loaded


In [14]:
# Load TATASTEEL 5-min data
data_path = Path("../data/historical/intraday_5min/TATASTEEL.parquet")
df = pd.read_parquet(data_path)

print(f"📊 Loaded: {len(df):,} candles")
print(f"📅 Period: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"\n🔍 First few rows:")
display(df.head(10))

📊 Loaded: 74,039 candles
📅 Period: 2022-01-03 09:15:00+05:30 to 2025-12-31 15:25:00+05:30

🔍 First few rows:


,datetime,open,high,low,close,volume,oi
0,2022-01-03 09:15:00+05:30,107.90,108.40,107.90,108.30,1910530,0
1,2022-01-03 09:20:00+05:30,108.30,108.40,108.10,108.15,857150,0
2,2022-01-03 09:25:00+05:30,108.20,108.35,108.10,108.25,460980,0
3,2022-01-03 09:30:00+05:30,108.25,108.30,108.00,108.15,593440,0
4,2022-01-03 09:35:00+05:30,108.15,108.35,108.05,108.35,405160,0
5,2022-01-03 09:40:00+05:30,108.35,108.60,108.25,108.55,935070,0
6,2022-01-03 09:45:00+05:30,108.55,108.70,108.50,108.65,720200,0
7,2022-01-03 09:50:00+05:30,108.65,108.85,108.60,108.85,899520,0
8,2022-01-03 09:55:00+05:30,108.85,108.90,108.60,108.70,790940,0
9,2022-01-03 10:00:00+05:30,108.70,108.75,108.60,108.65,302360,0


In [15]:
#Calculate MA20 on close price
df['ma20'] = df['close'].rolling(window=20).mean()

print("✅ MA20 calculated")
print(f"📊 First 25 rows (MA20 starts at row 20):")
display(df[['datetime', 'close', 'ma20']].head(25))

# Check for NaN values
print(f"\n🔍 Nan count in MA20: {df['ma20'].isna().sum()}")
print(f"   (First 19 rows will be NaN - this is expected)")

✅ MA20 calculated
📊 First 25 rows (MA20 starts at row 20):


,datetime,close,ma20
0,2022-01-03 09:15:00+05:30,108.30,NaN
1,2022-01-03 09:20:00+05:30,108.15,NaN
2,2022-01-03 09:25:00+05:30,108.25,NaN
3,2022-01-03 09:30:00+05:30,108.15,NaN
4,2022-01-03 09:35:00+05:30,108.35,NaN
5,2022-01-03 09:40:00+05:30,108.55,NaN
6,2022-01-03 09:45:00+05:30,108.65,NaN
7,2022-01-03 09:50:00+05:30,108.85,NaN
8,2022-01-03 09:55:00+05:30,108.70,NaN
9,2022-01-03 10:00:00+05:30,108.65,NaN



🔍 Nan count in MA20: 19
   (First 19 rows will be NaN - this is expected)


In [16]:
# Calculate True Range components
df['high_low'] =df['high'] - df['low']
df['high_prev_close'] = abs((df['high']) - df['close'].shift(1))
df['low_prev_close'] = abs((df['low']) - df['close'].shift(1))

# True Range = max of the three components
df['tr'] = df[['high_low', 'high_prev_close', 'low_prev_close']].max(axis=1)

# ATR = 14-period moving average of TR
df['atr'] = df['tr'].rolling(window=14).mean()

print("✅ ATR calculated")
print("📊 First 20 rows:")
display(df[['datetime', 'close', 'high_low', 'high_prev_close', 'low_prev_close', 'tr', 'atr']].head(20))

print(f"\n🔍 NaN count in ATR: {df['atr'].isna().sum()}")
print(f"   (First 14 rows will be NaN - this is expected)")


✅ ATR calculated
📊 First 20 rows:


,datetime,close,high_low,high_prev_close,low_prev_close,tr,atr
0,2022-01-03 09:15:00+05:30,108.30,0.50,NaN,NaN,0.50,NaN
1,2022-01-03 09:20:00+05:30,108.15,0.30,0.10,0.20,0.30,NaN
2,2022-01-03 09:25:00+05:30,108.25,0.25,0.20,0.05,0.25,NaN
3,2022-01-03 09:30:00+05:30,108.15,0.30,0.05,0.25,0.30,NaN
4,2022-01-03 09:35:00+05:30,108.35,0.30,0.20,0.10,0.30,NaN
5,2022-01-03 09:40:00+05:30,108.55,0.35,0.25,0.10,0.35,NaN
6,2022-01-03 09:45:00+05:30,108.65,0.20,0.15,0.05,0.20,NaN
7,2022-01-03 09:50:00+05:30,108.85,0.25,0.20,0.05,0.25,NaN
8,2022-01-03 09:55:00+05:30,108.70,0.30,0.05,0.25,0.30,NaN
9,2022-01-03 10:00:00+05:30,108.65,0.15,0.05,0.10,0.15,NaN



🔍 NaN count in ATR: 13
   (First 14 rows will be NaN - this is expected)


In [17]:
# Load TATASTEEL daily data
daily_path = Path("../data/historical/daily/TATASTEEL.parquet")
df_daily = pd.read_parquet(daily_path)

# Calculate MA50 and MA200 on daily closes
df_daily['ma50'] = df_daily['close'].rolling(window=50).mean()
df_daily['ma200'] = df_daily['close'].rolling(window=200).mean()

# Regime: Bullish if price > MA50 > MA200
df_daily['regime'] = 'neutral'
df_daily.loc[(df_daily['close'] > df_daily['ma50']) & (df_daily['ma50'] > df_daily['ma200']), 'regime'] = 'bullish'
df_daily.loc[(df_daily['close'] < df_daily['ma50']) & (df_daily['ma50'] < df_daily['ma200']), 'regime'] = 'bearish'

print(f"✅ Daily data loaded: {len(df_daily)} candles")
print(f"📅 Period: {df_daily['datetime'].min()} to {df_daily['datetime'].max()}")
print(f"\n📊 Regime distribution:")
print(df_daily['regime'].value_counts())
display(df_daily[['datetime', 'close', 'ma50', 'ma200', 'regime']].tail(10))

✅ Daily data loaded: 992 candles
📅 Period: 2022-01-03 00:00:00+05:30 to 2025-12-31 00:00:00+05:30

📊 Regime distribution:
regime
neutral    488
bullish    381
bearish    123
Name: count, dtype: int64


,datetime,close,ma50,ma200,regime
982,2025-12-17 00:00:00+05:30,170.34,172.9020,160.30545,neutral
983,2025-12-18 00:00:00+05:30,168.12,172.8358,160.46070,neutral
984,2025-12-19 00:00:00+05:30,168.69,172.7708,160.61070,neutral
985,2025-12-22 00:00:00+05:30,169.22,172.6268,160.77080,neutral
986,2025-12-23 00:00:00+05:30,170.90,172.5676,160.93245,neutral
987,2025-12-24 00:00:00+05:30,170.07,172.5084,161.08610,neutral
988,2025-12-26 00:00:00+05:30,169.12,172.4806,161.20120,neutral
989,2025-12-29 00:00:00+05:30,172.30,172.4618,161.31050,neutral
990,2025-12-30 00:00:00+05:30,175.80,172.4978,161.43170,bullish
991,2025-12-31 00:00:00+05:30,180.08,172.6550,161.57685,bullish


In [18]:
# Prepare daily data for merge - keep only date and regime
df_daily['date'] = df_daily['datetime'].dt.date
df_regime = df_daily[['date', 'regime']].copy()

# Add date column to intraday data
df['date'] = df['datetime'].dt.date

# Merge regime into intraday data
df = df.merge(df_regime, on='date', how='left')

print("✅ Regime merged into intraday data")
print(f"📊 Sample with regime:")
display(df[['datetime', 'date', 'close', 'ma20', 'atr', 'regime']].head(30))

print(f"\n🔍 Regime distribution in intraday data:")
print(df['regime'].value_counts())

✅ Regime merged into intraday data
📊 Sample with regime:


,datetime,date,close,ma20,atr,regime
0,2022-01-03 09:15:00+05:30,2022-01-03,108.30,NaN,NaN,neutral
1,2022-01-03 09:20:00+05:30,2022-01-03,108.15,NaN,NaN,neutral
2,2022-01-03 09:25:00+05:30,2022-01-03,108.25,NaN,NaN,neutral
3,2022-01-03 09:30:00+05:30,2022-01-03,108.15,NaN,NaN,neutral
4,2022-01-03 09:35:00+05:30,2022-01-03,108.35,NaN,NaN,neutral
5,2022-01-03 09:40:00+05:30,2022-01-03,108.55,NaN,NaN,neutral
6,2022-01-03 09:45:00+05:30,2022-01-03,108.65,NaN,NaN,neutral
7,2022-01-03 09:50:00+05:30,2022-01-03,108.85,NaN,NaN,neutral
8,2022-01-03 09:55:00+05:30,2022-01-03,108.70,NaN,NaN,neutral
9,2022-01-03 10:00:00+05:30,2022-01-03,108.65,NaN,NaN,neutral



🔍 Regime distribution in intraday data:
regime
neutral    36536
bullish    28404
bearish     9099
Name: count, dtype: int64


In [19]:
print("=" * 70)
print("✅ INDICATOR CALCULATION COMPLETE")
print("=" * 70)

print(f"\n📊 Data Summary:")
print(f"   Total candles: {len(df):,}")
print(f"   Period: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"   Columns: {', '.join(df.columns)}")

print(f"\n📈 Indicator Status:")
print(f"   MA20 NaN count: {df['ma20'].isna().sum()} (first 19 expected)")
print(f"   ATR NaN count: {df['atr'].isna().sum()} (first 14 expected)")
print(f"   Regime NaN count: {df['regime'].isna().sum()}")

print(f"\n🎯 Next Steps:")
print(f"   1. Build 03_bounce_detection.ipynb")
print(f"   2. Detect MA20 bounces using these indicators")
print(f"   3. Apply entry/exit logic")

display(df[['date']].head(10))

✅ INDICATOR CALCULATION COMPLETE

📊 Data Summary:
   Total candles: 74,039
   Period: 2022-01-03 09:15:00+05:30 to 2025-12-31 15:25:00+05:30
   Columns: datetime, open, high, low, close, volume, oi, ma20, high_low, high_prev_close, low_prev_close, tr, atr, date, regime

📈 Indicator Status:
   MA20 NaN count: 19 (first 19 expected)
   ATR NaN count: 13 (first 14 expected)
   Regime NaN count: 0

🎯 Next Steps:
   1. Build 03_bounce_detection.ipynb
   2. Detect MA20 bounces using these indicators
   3. Apply entry/exit logic


,date
0,2022-01-03
1,2022-01-03
2,2022-01-03
3,2022-01-03
4,2022-01-03
5,2022-01-03
6,2022-01-03
7,2022-01-03
8,2022-01-03
9,2022-01-03
